In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"],
    check=True,
)

print("Đã gỡ torchvision. Hãy restart session.")

In [ ]:
import base64
import importlib
import os
import subprocess
import sys
from pathlib import Path

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
hf_token = secrets.get_secret('HF_TOKEN')
github_token = secrets.get_secret('GITHUB_TOKEN')
if not hf_token or not github_token:
    raise RuntimeError('Thiếu Kaggle Secret HF_TOKEN hoặc GITHUB_TOKEN.')
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token

REPO_URL = 'https://github.com/huutamm1612/vieneu-ngoc-huyen-tts.git'
REPO_DIR = Path('/kaggle/working/vieneu-ngoc-huyen-tts')
if not REPO_DIR.exists():
    credential = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
    subprocess.run(
        [
            'git', '-c', f'http.extraHeader=Authorization: Basic {credential}',
            'clone', '--depth', '1', REPO_URL, str(REPO_DIR),
        ],
        check=True,
    )
if not (REPO_DIR / 'pyproject.toml').is_file():
    raise RuntimeError(f'Repository không hợp lệ: {REPO_DIR}')
SOURCE_DIR = REPO_DIR / 'src'
INFERENCE_INIT = SOURCE_DIR / 'inference' / '__init__.py'
if not INFERENCE_INIT.is_file():
    raise RuntimeError(
        f'Repository chưa có src/inference: {REPO_DIR}. '
        'Hãy cập nhật GitHub repo rồi tạo Kaggle session mới.'
    )
source_value = str(SOURCE_DIR.resolve())
if source_value not in sys.path:
    sys.path.insert(0, source_value)
importlib.invalidate_caches()
print('Repo:', REPO_DIR)
print('Source:', SOURCE_DIR)
print('Python:', sys.version.split()[0])

In [ ]:
# Chỉ cần chạy một lần cho mỗi Kaggle session. Nếu Kaggle yêu cầu restart sau khi
# đổi Torch, restart session rồi Run All lại từ cell đầu tiên.
# VieNeu-TTS không dùng torchvision. Gỡ bản có sẵn của Kaggle để tránh binary
# torchvision cũ xung đột với torch==2.8.0 do project cài đặt.
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision'],
    check=True,
)
subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '-q',
        'ipywidgets>=8,<9', '-e', f'{REPO_DIR}[inference]',
    ],
    check=True,
)
importlib.invalidate_caches()
import inference
print('Đã cài inference package và widget upload TXT.')
print('Inference package:', inference.__file__)

In [ ]:
from huggingface_hub import snapshot_download

MODEL_REPO_ID = 'YOUR_HF_USERNAME/vieneu-tts-ngoc-huyen'
MODEL_REVISION = 'main'

if MODEL_REPO_ID.startswith('YOUR_') or '/' not in MODEL_REPO_ID:
    raise ValueError('Hãy thay MODEL_REPO_ID bằng username/model-name trên Hugging Face.')

model_name = MODEL_REPO_ID.split('/', 1)[1]
MODEL_DOWNLOAD_DIR = Path('/kaggle/working/hf_models') / model_name
MODEL_PATH = Path(
    snapshot_download(
        repo_id=MODEL_REPO_ID,
        repo_type='model',
        revision=MODEL_REVISION,
        local_dir=MODEL_DOWNLOAD_DIR,
        token=hf_token,
    )
)

required_model_files = ['COMPLETE.json', 'config.json', 'tokenizer.json']
missing_model_files = [name for name in required_model_files if not (MODEL_PATH / name).is_file()]
weight_files = list(MODEL_PATH.glob('model*.safetensors'))
if missing_model_files or not weight_files:
    raise FileNotFoundError(
        f'Model tải về chưa đầy đủ; missing={missing_model_files}, weights={len(weight_files)}'
    )
print('Model:', MODEL_REPO_ID, '@', MODEL_REVISION)
print('Local path:', MODEL_PATH)
print('Weight files:', [path.name for path in weight_files])

In [ ]:
import ipywidgets as widgets
import torch
from IPython.display import display

REF_AUDIO_PATH = Path(
    'reference.wav'
)
REF_TEXT = (
    'Transcript phải khớp chính xác với reference audio.'
)

TEST_MIN_CHARS = 80
TEST_TARGET_CHARS = 128
TEST_MAX_CHARS = 156
TEST_BATCH_SIZE = 128
TEST_MAX_LENGTH_GAP = 12

GPU_COUNT = torch.cuda.device_count()
NUM_GPUS = min(2, GPU_COUNT) if GPU_COUNT else 1
GPU_NAMES = [torch.cuda.get_device_name(index) for index in range(GPU_COUNT)]
MAX_RUNTIME_BATCH_SIZE = 8 if any('T4' in name for name in GPU_NAMES) else None

if not REF_AUDIO_PATH.is_file():
    raise FileNotFoundError(f'Không tìm thấy reference audio: {REF_AUDIO_PATH}')

def get_uploaded_file(uploader):
    value = uploader.value
    if not value:
        return None
    if isinstance(value, dict):
        filename, file_info = next(iter(value.items()))
    else:
        file_info = value[0]
        filename = file_info['name']
    content = file_info['content']
    content = content.tobytes() if isinstance(content, memoryview) else bytes(content)
    return Path(filename).name, content

def save_uploaded_txt(uploader):
    uploaded = get_uploaded_file(uploader)
    if uploaded is None:
        raise ValueError('Hãy bấm Chọn TXT và upload file trước khi chạy cell này.')
    filename, content = uploaded
    if Path(filename).suffix.lower() != '.txt':
        raise ValueError(f'File phải có đuôi .txt: {filename}')
    if not content:
        raise ValueError(f'File TXT rỗng: {filename}')
    upload_dir = Path('/kaggle/working/uploaded_txt')
    upload_dir.mkdir(parents=True, exist_ok=True)
    destination = upload_dir / filename
    destination.write_bytes(content)
    return destination

txt_uploader = widgets.FileUpload(
    accept='.txt',
    multiple=False,
    description='Chọn TXT',
)
display(txt_uploader)
print('GPU:', GPU_NAMES or ['CPU'])
print('Sau khi chọn file, chạy cell chuẩn bị batches bên dưới.')

In [ ]:
from inference import InferenceConfig, TTSInference, prepare_batches

INPUT_TXT = save_uploaded_txt(txt_uploader)
OUTPUT_WAV = Path('/kaggle/working') / f'{INPUT_TXT.stem}.wav'

batches = prepare_batches(
    input_path=INPUT_TXT,
    min_chars=TEST_MIN_CHARS,
    target_chars=TEST_TARGET_CHARS,
    max_chars=TEST_MAX_CHARS,
    batch_size=TEST_BATCH_SIZE,
    max_length_gap=TEST_MAX_LENGTH_GAP,
)
items = [item for batch in batches for item in batch]
print('TXT:', INPUT_TXT)
print(f'{len(items)} đoạn trong {len(batches)} logical batches')
print('Đoạn đầu:', min(items, key=lambda item: item['index'])['text'])
print('Output:', OUTPUT_WAV)

In [ ]:
config = InferenceConfig(
    model=str(MODEL_PATH),
    devices='auto',
    num_gpus=NUM_GPUS,
    min_chars=TEST_MIN_CHARS,
    target_chars=TEST_TARGET_CHARS,
    max_chars=TEST_MAX_CHARS,
    batch_size=TEST_BATCH_SIZE,
    max_length_gap=TEST_MAX_LENGTH_GAP,
    max_runtime_batch_size=MAX_RUNTIME_BATCH_SIZE,
    do_sample=False,
    keep_segments=False,
    show_progress=True,
)

with TTSInference(config) as tts:
    result = tts.infer(
        batches=batches,
        reference_audio=REF_AUDIO_PATH,
        reference_text=REF_TEXT,
        output_path=OUTPUT_WAV,
    )

print(result.as_dict()["output_path"])

In [ ]:
from IPython.display import Audio, FileLink, display

display(Audio(filename=str(OUTPUT_WAV)))
display(FileLink(str(OUTPUT_WAV)))